
# IFEST DAC 2026 — V3-A Surgical Recomposition

**Agreed design only. No new Transformer training.**

V3-A surgically recombines the two already-trained systems:

- **45% V1 cross-fitted meta probability**
- **55% V2 cross-fitted meta probability**
- threshold selected **only from blended strict OOF**
- **V2 graph package frozen**: exact pair → unique canonical-content mismatch → confidence-gated same-title mismatch → semantic fallback

This notebook intentionally does **not** retrain mDeBERTa, alter class weights, add a backbone, retune graph heuristics from leaderboard feedback, or introduce synthetic experiments. This is the resource-efficient implementation of the agreed V3-A.

The prior V1/V2 OOF and test-score artifacts are embedded as frozen experiment outputs so the notebook can run in Kaggle without requiring the previous result folders as additional inputs. The competition `train.csv`, `test.csv`, and `sample_submission.csv` are still read from Kaggle Input and verified before any output is produced.


In [13]:

import os, re, io, gc, json, gzip, base64, hashlib, unicodedata, warnings, zipfile
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from sklearn.model_selection import StratifiedGroupKFold

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 180)

# ---------------- V3-A FROZEN CONFIG ----------------
SEED = 2026
N_FOLDS = 3
V1_WEIGHT = 0.45
V2_WEIGHT = 0.55

# Frozen V2 graph package. These are not retuned in V3.
V2_TITLE_GATE = 0.9839511586057019
MIN_GRAPH_CONTENT_WORDS = 20

PUBLIC_LB_V1 = 0.93367
PUBLIC_LB_V2 = 0.94856

EXPECTED_TRAIN_ROWS = 14397
EXPECTED_TEST_ROWS = 3603
EXPECTED_DATA_HASHES = {
    "train": "d2c3cf75cb6ef8d502f32d9b8ed8b22508040ea8570a295269a53b67e1e184ab",
    "test": "0871476c2c74ee5ba6daa931dbd6cfe1d3a992ac3da9c4ea5228c1deed16cf88",
    "sample_submission": "2725f8fe1c074b5e1d7c7f1234cfa482871d2ea94f6056b4451c22552809ebf5",
}

OUTPUT_DIR = Path("/kaggle/working" if Path("/kaggle/working").exists() else "/mnt/data/ifest_v3_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("IFEST DAC V3-A Surgical Recomposition")
print(f"Weights: V1={V1_WEIGHT:.2f}, V2={V2_WEIGHT:.2f}")
print("Training: intentionally disabled by design (recomposition-only)")
try:
    import torch
    print(f"PyTorch={torch.__version__} | CUDA={torch.cuda.is_available()} | GPUs={torch.cuda.device_count()}")
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            print(i, torch.cuda.get_device_name(i))
except Exception as e:
    print("Torch diagnostic unavailable:", repr(e))


IFEST DAC V3-A Surgical Recomposition
Weights: V1=0.45, V2=0.55
Training: intentionally disabled by design (recomposition-only)
PyTorch=2.10.0+cu128 | CUDA=True | GPUs=2
0 Tesla T4
1 Tesla T4


In [14]:
FROZEN_ARTIFACTS = {
    'V1_OOF': {
        'sha256': '6de4208cdeec90ad5e589155d93e9613f02d5dd7f82c7b88c8f7712c371026e0',
        'rows': 14397,
        'columns': ['id', 'label', 'meta_prob'],
        'payload': (
            '...'
        ),
    },
    'V1_TEST': {
        'sha256': 'f8c4b434ddb482d8d7669cbcb3a7ab35296153f60399dbe1654c5d174ade66c4',
        'rows': 3603,
        'columns': ['id', 'meta_prob', 'label'],
        'payload': (
            '...'
        ),
    },
    'V2_OOF': {
        'sha256': 'e4b81b9cbbd6d6c5d7c101151f693c9383e80ba3be1f82c97880d4dc1be6805a',
        'rows': 14397,
        'columns': ['id', 'label', 'meta_prob'],
        'payload': (
            '...'
        ),
    },
    'V2_TEST': {
        'sha256': 'b23ded13305fc1ec0e726609ef58c7193fbf929559ca5b883f07b4b765266d26',
        'rows': 3603,
        'columns': ['id', 'meta_prob', 'graph_rule', 'forced_label', 'label'],
        'payload': (
            '...'
        ),
    },
}


def decode_frozen_artifact(spec):
    raw = gzip.decompress(base64.b64decode(spec["payload"].encode()))
    got = hashlib.sha256(raw).hexdigest()
    assert got == spec["sha256"], f"Frozen artifact checksum mismatch: {got} != {spec['sha256']}"
    df = pd.read_csv(io.BytesIO(raw))
    assert len(df) == spec["rows"]
    assert df.columns.tolist() == spec["columns"]
    return df

v1_oof = decode_frozen_artifact(FROZEN_ARTIFACTS["V1_OOF"])
v1_test = decode_frozen_artifact(FROZEN_ARTIFACTS["V1_TEST"])
v2_oof = decode_frozen_artifact(FROZEN_ARTIFACTS["V2_OOF"])
v2_test = decode_frozen_artifact(FROZEN_ARTIFACTS["V2_TEST"])
print("Frozen artifacts decoded and checksum-verified:")
print(" V1 OOF", v1_oof.shape, "| V1 test", v1_test.shape)
print(" V2 OOF", v2_oof.shape, "| V2 test", v2_test.shape)


Frozen artifacts decoded and checksum-verified:
 V1 OOF (14397, 3) | V1 test (3603, 3)
 V2 OOF (14397, 3) | V2 test (3603, 5)


In [15]:

def find_competition_csv(filename):
    roots = [Path("/kaggle/input"), Path("/mnt/data/ifest_dac")]
    candidates = []
    for root in roots:
        if not root.exists():
            continue
        if root.is_file():
            continue
        candidates.extend(root.rglob(filename))
    # Prefer an IFEST/competition-looking path, then the shortest deterministic path.
    candidates = sorted(set(candidates), key=lambda p: (0 if "ifest" in str(p).lower() else 1, len(str(p)), str(p)))
    if not candidates:
        raise FileNotFoundError(f"Could not find {filename} under /kaggle/input. Add the IFEST competition dataset as a Kaggle Input.")
    return candidates[0]

train_path = find_competition_csv("train.csv")
test_path = find_competition_csv("test.csv")
sample_path = find_competition_csv("sample_submission.csv")

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample = pd.read_csv(sample_path)

print("train:", train.shape, train_path)
print("test :", test.shape, test_path)
print("sample:", sample.shape, sample_path)

assert train.shape == (EXPECTED_TRAIN_ROWS, 4), f"Unexpected train shape: {train.shape}"
assert test.shape == (EXPECTED_TEST_ROWS, 3), f"Unexpected test shape: {test.shape}"
assert {"id","title","content","label"}.issubset(train.columns)
assert {"id","title","content"}.issubset(test.columns)
assert sample.columns.tolist() == ["id","label"]
assert sample.id.astype(str).tolist() == test.id.astype(str).tolist()

# Logical CSV fingerprint protects the frozen prior outputs from accidental dataset mismatch.
def logical_df_hash(df):
    raw = df.to_csv(index=False, lineterminator="\n").encode()
    return hashlib.sha256(raw).hexdigest()

for name, df in [("train", train), ("test", test), ("sample_submission", sample)]:
    got = logical_df_hash(df)
    exp = EXPECTED_DATA_HASHES[name]
    print(f"{name} hash:", got)
    assert got == exp, f"{name} does not match the dataset used for V1/V2. Expected {exp}, got {got}"

# Alignment of frozen model outputs with this exact competition dataset.
assert v1_oof.id.astype(str).tolist() == train.id.astype(str).tolist()
assert v2_oof.id.astype(str).tolist() == train.id.astype(str).tolist()
assert v1_oof.label.astype(int).tolist() == train.label.astype(int).tolist()
assert v2_oof.label.astype(int).tolist() == train.label.astype(int).tolist()
assert v1_test.id.astype(str).tolist() == test.id.astype(str).tolist()
assert v2_test.id.astype(str).tolist() == test.id.astype(str).tolist()
print("Dataset + frozen-output alignment: OK")


train: (14397, 4) /kaggle/input/competitions/penyisihan-ifest-2026-dac/train.csv
test : (3603, 3) /kaggle/input/competitions/penyisihan-ifest-2026-dac/test.csv
sample: (3603, 2) /kaggle/input/competitions/penyisihan-ifest-2026-dac/sample_submission.csv
train hash: d2c3cf75cb6ef8d502f32d9b8ed8b22508040ea8570a295269a53b67e1e184ab
test hash: 0871476c2c74ee5ba6daa931dbd6cfe1d3a992ac3da9c4ea5228c1deed16cf88
sample_submission hash: 2725f8fe1c074b5e1d7c7f1234cfa482871d2ea94f6056b4451c22552809ebf5
Dataset + frozen-output alignment: OK



## Strict OOF recomposition

The blend weight is **frozen at 0.45 / 0.55** from the prior audit. Only the binary threshold is selected from the blended OOF labels, which is the agreed V3-A validation procedure.


In [16]:

def macro_f1_parts(y, pred):
    per = f1_score(y, pred, labels=[0,1], average=None, zero_division=0)
    return float(per.mean()), float(per[0]), float(per[1])


def best_exact_macro_f1_threshold(y_true, probs):
    """Exact O(n log n) threshold sweep for predict-1 when p >= threshold."""
    y = np.asarray(y_true, dtype=np.int8)
    p = np.asarray(probs, dtype=np.float64)
    assert len(y) == len(p) and np.isfinite(p).all()

    order = np.argsort(p, kind="mergesort")
    ys, ps = y[order], p[order]
    n0, n1 = int((y == 0).sum()), int((y == 1).sum())

    # threshold below min: all predicted 1
    tp1, fp1, fn1 = n1, n0, 0
    tp0, fp0, fn0 = 0, 0, n0

    def score():
        d1 = 2*tp1 + fp1 + fn1
        d0 = 2*tp0 + fp0 + fn0
        f1_1 = 0.0 if d1 == 0 else 2*tp1/d1
        f1_0 = 0.0 if d0 == 0 else 2*tp0/d0
        return (f1_0 + f1_1)/2, f1_0, f1_1

    best_macro, best_f0, best_f1 = score()
    best_t = float(np.nextafter(ps[0], -np.inf))

    i = 0
    while i < len(y):
        val = ps[i]
        j = i
        while j < len(y) and ps[j] == val:
            if ys[j] == 1:
                tp1 -= 1; fn1 += 1; fp0 += 1
            else:
                fp1 -= 1; tp0 += 1; fn0 -= 1
            j += 1
        m, f0, f1 = score()
        t = float(np.nextafter(val, np.inf))
        if m > best_macro + 1e-15:
            best_macro, best_f0, best_f1, best_t = m, f0, f1, t
        i = j
    return {"threshold": best_t, "macro_f1": float(best_macro), "f1_0": float(best_f0), "f1_1": float(best_f1)}


y = train.label.to_numpy(dtype=np.int8)
v1_oof_prob = v1_oof.meta_prob.to_numpy(dtype=float)
v2_oof_prob = v2_oof.meta_prob.to_numpy(dtype=float)
blend_oof = V1_WEIGHT * v1_oof_prob + V2_WEIGHT * v2_oof_prob

v1_diag = best_exact_macro_f1_threshold(y, v1_oof_prob)
v2_diag = best_exact_macro_f1_threshold(y, v2_oof_prob)
v3_diag = best_exact_macro_f1_threshold(y, blend_oof)
V3_THRESHOLD = v3_diag["threshold"]
v3_oof_label = (blend_oof >= V3_THRESHOLD).astype(np.int8)

summary = pd.DataFrame([
    {"system":"V1 meta OOF", **v1_diag},
    {"system":"V2 meta OOF", **v2_diag},
    {"system":"V3 45/55 blend OOF", **v3_diag},
])
display(summary)
print("V3 confusion matrix:\n", confusion_matrix(y, v3_oof_label))
print(classification_report(y, v3_oof_label, digits=5, zero_division=0))

assert v3_diag["macro_f1"] > max(v1_diag["macro_f1"], v2_diag["macro_f1"]), "V3 blend no longer beats both frozen OOF systems; aborting submission generation."


,system,threshold,macro_f1,f1_0,f1_1
0,V1 meta OOF,0.652796,0.840644,0.712513,0.968775
1,V2 meta OOF,0.596939,0.836466,0.704748,0.968184
2,V3 45/55 blend OOF,0.581718,0.849301,0.727273,0.971330


V3 confusion matrix:
 [[  996   441]
 [  306 12654]]
              precision    recall  f1-score   support

           0    0.76498   0.69311   0.72727      1437
           1    0.96632   0.97639   0.97133     12960

    accuracy                        0.94811     14397
   macro avg    0.86565   0.83475   0.84930     14397
weighted avg    0.94623   0.94811   0.94697     14397



In [17]:

# Reconstruct the same connected-component groups used in V1/V2 for transparent fold stability diagnostics.
TOKEN_RE = re.compile(r"[\w]+", flags=re.UNICODE)
SOURCE_SUFFIX_RE = re.compile(r"\s+kumparan\s+com\s*$", re.I)

def clean_ws(text):
    text = unicodedata.normalize("NFKC", str(text)).replace("\xa0", " ")
    return re.sub(r"\s+", " ", text).strip()

def clean_title_for_model(text):
    return SOURCE_SUFFIX_RE.sub("", clean_ws(text)).strip()

def normalize_key(text):
    return " ".join(TOKEN_RE.findall(clean_ws(text).lower()))

for df in (train, test):
    df["title_model"] = df.title.map(clean_title_for_model)
    df["title_key"] = df.title_model.map(normalize_key)
    df["content_key"] = df.content.map(normalize_key)
    df["pair_key"] = df.title_key + "|||" + df.content_key
    df["content_words"] = df.content.map(lambda x: len(clean_ws(x).split()))

class UnionFind:
    def __init__(self, n):
        self.p = list(range(n)); self.r = [0] * n
    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]; x = self.p[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb: return
        if self.r[ra] < self.r[rb]: ra, rb = rb, ra
        self.p[rb] = ra
        if self.r[ra] == self.r[rb]: self.r[ra] += 1

def build_component_groups(df):
    uf = UnionFind(len(df))
    for col in ["title_key", "content_key"]:
        first = {}
        for i, key in enumerate(df[col].tolist()):
            if key in first: uf.union(i, first[key])
            else: first[key] = i
    roots = [uf.find(i) for i in range(len(df))]
    mapping = {r:j for j,r in enumerate(dict.fromkeys(roots))}
    return np.asarray([mapping[r] for r in roots], dtype=np.int32)

groups = build_component_groups(train)
cv = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
splits = list(cv.split(train, y, groups=groups))
fold_rows = []
for fold, (tr_idx, va_idx) in enumerate(splits):
    assert set(groups[tr_idx]).isdisjoint(set(groups[va_idx]))
    pred = (blend_oof[va_idx] >= V3_THRESHOLD).astype(np.int8)
    m,f0,f1 = macro_f1_parts(y[va_idx], pred)
    fold_rows.append({"fold":fold,"valid_rows":len(va_idx),"neg_rate":float((y[va_idx]==0).mean()),"macro_f1":m,"f1_0":f0,"f1_1":f1})
fold_metrics = pd.DataFrame(fold_rows)
print("Connected components:", len(np.unique(groups)))
print("Rows in non-singleton components:", int(pd.Series(groups).value_counts().loc[lambda s:s>1].sum()))
display(fold_metrics)


Connected components: 11525
Rows in non-singleton components: 5419


,fold,valid_rows,neg_rate,macro_f1,f1_0,f1_1
0,0,4798,0.099416,0.859613,0.746204,0.973023
1,1,4807,0.095694,0.832263,0.694954,0.969572
2,2,4792,0.104341,0.855017,0.738624,0.971409



## Frozen V2 graph package parity check

V3 does **not** retune graph rules. The graph decisions are reconstructed from the competition train/test text using the frozen V2 logic and **V2 meta probability for the gated-title condition**. An assertion then verifies exact parity with the V2 run artifact before the graph is allowed to override the V3 blended semantic decision.


In [18]:

def build_graph_reference(ref_df):
    pair_to_label = {}
    for key, g in ref_df.groupby("pair_key"):
        labs = set(map(int, g.label))
        if len(labs) == 1:
            pair_to_label[key] = next(iter(labs))
    content_to_pos_titles = defaultdict(set)
    title_to_pos_contents = defaultdict(set)
    for r in ref_df.loc[ref_df.label == 1].itertuples():
        content_to_pos_titles[r.content_key].add(r.title_key)
        title_to_pos_contents[r.title_key].add(r.content_key)
    return pair_to_label, content_to_pos_titles, title_to_pos_contents


def frozen_v2_graph_decisions(query_df, ref_df, v2_meta_prob):
    pair_to_label, content_to_pos_titles, title_to_pos_contents = build_graph_reference(ref_df)
    rules = np.array(["semantic"] * len(query_df), dtype=object)
    forced = np.full(len(query_df), -1, dtype=np.int8)
    v2_meta_prob = np.asarray(v2_meta_prob, dtype=float)

    for pos, r in enumerate(query_df.itertuples()):
        # 1) exact pair
        if r.pair_key in pair_to_label:
            forced[pos] = int(pair_to_label[r.pair_key])
            rules[pos] = "exact_pair"
            continue

        malformed = int(r.content_words) < MIN_GRAPH_CONTENT_WORDS
        pos_titles = content_to_pos_titles.get(r.content_key, set())

        # 2) unique canonical positive-content mismatch
        if not malformed and len(pos_titles) == 1 and r.title_key not in pos_titles:
            forced[pos] = 0
            rules[pos] = "unique_content_different_title"
            continue

        # 3) frozen V2 gated same-title mismatch; gating stays on V2 meta scale.
        pos_contents = title_to_pos_contents.get(r.title_key, set())
        if (
            not malformed and pos_contents and r.content_key not in pos_contents
            and v2_meta_prob[pos] < V2_TITLE_GATE
        ):
            forced[pos] = 0
            rules[pos] = "gated_title_different_content"

    return rules, forced

v2_test_prob = v2_test.meta_prob.to_numpy(dtype=float)
reconstructed_rule, reconstructed_forced = frozen_v2_graph_decisions(test, train, v2_test_prob)

# Frozen-package integrity: reconstructed rules must exactly equal the already-submitted V2 artifact.
assert np.array_equal(reconstructed_rule, v2_test.graph_rule.astype(str).to_numpy()), "V2 graph rule parity failure"
assert np.array_equal(reconstructed_forced, v2_test.forced_label.to_numpy(dtype=np.int8)), "V2 forced-label parity failure"
print("Frozen V2 graph parity: EXACT MATCH")
display(pd.Series(reconstructed_rule).value_counts().to_frame("rows"))


Frozen V2 graph parity: EXACT MATCH


,rows
semantic,2447
exact_pair,927
unique_content_different_title,212
gated_title_different_content,17


In [19]:

# ---------------- V3 TEST INFERENCE ----------------
v1_test_prob = v1_test.meta_prob.to_numpy(dtype=float)
v2_test_prob = v2_test.meta_prob.to_numpy(dtype=float)
blend_test = V1_WEIGHT * v1_test_prob + V2_WEIGHT * v2_test_prob

semantic_v3_label = (blend_test >= V3_THRESHOLD).astype(np.int8)
final_v3_prob = blend_test.copy()
final_v3_label = semantic_v3_label.copy()

mask_forced = reconstructed_forced >= 0
final_v3_label[mask_forced] = reconstructed_forced[mask_forced]
final_v3_prob[mask_forced] = reconstructed_forced[mask_forced].astype(float)

v1_final_label = v1_test.label.to_numpy(dtype=np.int8)
v2_final_label = v2_test.label.to_numpy(dtype=np.int8)

print("Prediction distributions")
print("V1:", pd.Series(v1_final_label).value_counts().sort_index().to_dict())
print("V2:", pd.Series(v2_final_label).value_counts().sort_index().to_dict())
print("V3:", pd.Series(final_v3_label).value_counts().sort_index().to_dict())
print("V3 changes vs V1:", int((final_v3_label != v1_final_label).sum()))
print("V3 changes vs V2:", int((final_v3_label != v2_final_label).sum()))
print("V3 graph-forced rows:", int(mask_forced.sum()))

# Pre-agreed sanity anchors from the audit. These guard against accidental implementation drift.
assert int((final_v3_label == 0).sum()) == 356, "Unexpected V3 negative count; methodology or input artifacts changed."
assert int((final_v3_label != v2_final_label).sum()) == 33, "Unexpected V3-vs-V2 delta count; aborting."
assert int((final_v3_label != v1_final_label).sum()) == 46, "Unexpected V3-vs-V1 delta count; aborting."


Prediction distributions
V1: {0: 368, 1: 3235}
V2: {0: 369, 1: 3234}
V3: {0: 356, 1: 3247}
V3 changes vs V1: 46
V3 changes vs V2: 33
V3 graph-forced rows: 1156


In [20]:

# Prediction-delta audit: only diagnostics, not used to tune the submission.
change_mask = (final_v3_label != v2_final_label) | (final_v3_label != v1_final_label)
changed = pd.DataFrame({
    "id": test.id,
    "title": test.title,
    "v1_meta_prob": v1_test_prob,
    "v2_meta_prob": v2_test_prob,
    "v3_blend_prob": blend_test,
    "graph_rule": reconstructed_rule,
    "forced_label": reconstructed_forced,
    "v1_label": v1_final_label,
    "v2_label": v2_final_label,
    "v3_label": final_v3_label,
}).loc[change_mask].reset_index(drop=True)
print("Rows changed from at least one previous version:", len(changed))
display(changed.head(30))


Rows changed from at least one previous version: 79


,id,title,v1_meta_prob,v2_meta_prob,v3_blend_prob,graph_rule,forced_label,v1_label,v2_label,v3_label
0,te00029,Bantu Cerdaskan Anak Perbatasan RI PNG TNI Ubah Truk Jadi Perpustakaan,0.358408,0.846756,0.627000,semantic,-1,0,1,1
1,te00084,Covid 19 di Jakarta Terus Melonjak PDI P Minta Anies Lihat Langsung Ketidakpatuhan Warga,0.650050,0.938590,0.808747,semantic,-1,0,1,1
2,te00100,Masih Ngegas Kasus Aktif Corona di Jateng Tambah 1 098 Hari Ini,0.484803,0.759378,0.635820,semantic,-1,0,1,1
3,te00119,Ridwan Kamil Dorong Masyarakat Tetap Produktif Saat Pandemi Asal,0.934645,0.368875,0.623471,semantic,-1,1,0,1
4,te00142,Mengenal Vaksin Sinovac yang Dipakai RI Basmi Covid 19,0.834809,0.553571,0.680129,semantic,-1,1,0,1
5,te00152,Kemenag 667 Pasien Positif Covid 19 Jalani Isolasi di Asrama Haji Seluruh Indonesia,0.636220,0.709380,0.676458,semantic,-1,0,1,1
6,te00172,Realisasi Anggaran Pemulihan Ekonomi Baru Rp286 3 Triliun,0.736893,0.523288,0.619410,semantic,-1,1,0,1
7,te00208,Menkes Singgung Omongan Bill Gates Bangun Sistem Senjata Luput Deteksi Virus kumparan com,0.913810,0.408560,0.635922,semantic,-1,1,0,1
8,te00287,Was was Janji Manis Vaksin Corona,0.722156,0.502807,0.601514,semantic,-1,1,0,1
9,te00320,KPK Saran Pengadaan Vaksin Corona Tak Langsung Jumlah Besar Kenapa,0.964868,0.965324,0.965119,gated_title_different_content,0,1,0,0


In [21]:

# ---------------- EXPORT ----------------
submission = sample[["id"]].copy()
submission["label"] = final_v3_label.astype(int)

submission_path = OUTPUT_DIR / "submission_v3.csv"
oof_path = OUTPUT_DIR / "oof_v3.csv"
test_scores_path = OUTPUT_DIR / "test_scores_v3.csv"
validation_path = OUTPUT_DIR / "validation_summary_v3.csv"
fold_path = OUTPUT_DIR / "fold_metrics_v3.csv"
changes_path = OUTPUT_DIR / "changed_rows_v3.csv"
config_path = OUTPUT_DIR / "config_v3.json"
zip_path = OUTPUT_DIR / "results_v3.zip"

submission.to_csv(submission_path, index=False)

pd.DataFrame({
    "id": train.id,
    "label": y.astype(int),
    "v1_meta_prob": v1_oof_prob,
    "v2_meta_prob": v2_oof_prob,
    "blend_prob": blend_oof,
    "blend_label": v3_oof_label.astype(int),
}).to_csv(oof_path, index=False)

pd.DataFrame({
    "id": test.id,
    "v1_meta_prob": v1_test_prob,
    "v2_meta_prob": v2_test_prob,
    "blend_prob": blend_test,
    "semantic_label": semantic_v3_label.astype(int),
    "final_prob": final_v3_prob,
    "graph_rule": reconstructed_rule,
    "forced_label": reconstructed_forced.astype(int),
    "label": final_v3_label.astype(int),
}).to_csv(test_scores_path, index=False)

summary.to_csv(validation_path, index=False)
fold_metrics.to_csv(fold_path, index=False)
changed.to_csv(changes_path, index=False)

config = {
    "version": "v3-a-surgical-recomposition",
    "seed": SEED,
    "n_folds": N_FOLDS,
    "training": False,
    "v1_weight": V1_WEIGHT,
    "v2_weight": V2_WEIGHT,
    "blend_threshold": float(V3_THRESHOLD),
    "strict_oof_macro_f1": float(v3_diag["macro_f1"]),
    "strict_oof_f1_0": float(v3_diag["f1_0"]),
    "strict_oof_f1_1": float(v3_diag["f1_1"]),
    "graph_package": "frozen-v2",
    "v2_title_gate": V2_TITLE_GATE,
    "min_graph_content_words": MIN_GRAPH_CONTENT_WORDS,
    "graph_forced_test_rows": int(mask_forced.sum()),
    "test_negative_count": int((final_v3_label == 0).sum()),
    "changes_vs_v1": int((final_v3_label != v1_final_label).sum()),
    "changes_vs_v2": int((final_v3_label != v2_final_label).sum()),
    "public_lb_reference": {"v1": PUBLIC_LB_V1, "v2": PUBLIC_LB_V2},
}
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

files = [submission_path, oof_path, test_scores_path, validation_path, fold_path, changes_path, config_path]
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in files:
        zf.write(p, arcname=p.name)

# Final submission integrity checks.
check = pd.read_csv(submission_path)
assert check.shape == (EXPECTED_TEST_ROWS, 2)
assert check.columns.tolist() == ["id", "label"]
assert check.id.astype(str).tolist() == test.id.astype(str).tolist()
assert set(check.label.unique()).issubset({0,1})
assert not check.isna().any().any()

print("Saved V3 artifacts:")
for p in files + [zip_path]:
    print(" -", p)
print("\nSUBMISSION READY:", submission_path)
print("Strict OOF Macro F1:", round(v3_diag["macro_f1"], 6), "| threshold:", V3_THRESHOLD)
print("Final label distribution:", check.label.value_counts().sort_index().to_dict())


Saved V3 artifacts:
 - /kaggle/working/submission_v3.csv
 - /kaggle/working/oof_v3.csv
 - /kaggle/working/test_scores_v3.csv
 - /kaggle/working/validation_summary_v3.csv
 - /kaggle/working/fold_metrics_v3.csv
 - /kaggle/working/changed_rows_v3.csv
 - /kaggle/working/config_v3.json
 - /kaggle/working/results_v3.zip

SUBMISSION READY: /kaggle/working/submission_v3.csv
Strict OOF Macro F1: 0.849301 | threshold: 0.5817181386596663
Final label distribution: {0: 356, 1: 3247}



## Expected V3-A anchors

If the competition data and frozen V1/V2 experiment artifacts are unchanged, the notebook should report approximately:

- strict blended OOF Macro F1: **0.849301**
- OOF F1 class 0: **0.727273**
- OOF F1 class 1: **0.971330**
- final test negatives: **356**
- changed labels vs V2: **33**
- changed labels vs V1: **46**

Any violation of the exact implementation-drift assertions aborts submission generation rather than silently producing a different experiment.
